In [12]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

#Hyperparameter Tuning (1)
#pre-process the data
df = pd.read_csv("insurance.csv")

#With logistic regression
y= df['death'] #Select target variable
X = df.drop('death',axis=1) #Select features
X = pd.get_dummies(X, drop_first=True) 

#Normalize the data
columns = X.columns #create index with column names (needed for last step)
scaler = MinMaxScaler() #initiate the scaler
X = scaler.fit_transform(X) #scale the data
X = pd.DataFrame(X,columns=columns) #turn back into a dataframe

In [13]:
#Hyperparameter Tuning (1)
#pre-process the data
df = pd.read_csv("insurance.csv")

#With logistic regression
y= df['charges']
X = df.drop('charges',axis=1) #Select features
X = pd.get_dummies(X, drop_first=True) 

#Normalize the data
columns = X.columns #create index with column names (needed for last step)
scaler = MinMaxScaler() #initiate the scaler
X = scaler.fit_transform(X) #scale the data
X = pd.DataFrame(X,columns=columns) #turn back into a dataframe

In [14]:
#Hyperparameter Tuning (2)
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Hyperparameter tuning
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 250, step=10),
        'max_depth': trial.suggest_int('max_depth', 10, 250, step=10),
        'min_samples_split': trial.suggest_int('min_samples_split', 10, 30, step=1),
        'max_features': trial.suggest_categorical('max_features', [None, 'sqrt', 'log2']),
        'random_state': 69
    }
    regressor = RandomForestRegressor(**params)
    score = cross_val_score(regressor, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1).mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)  # search across 100 different combinations

print("Best params:", study.best_params)

[I 2026-10-02 12:04:31,202] A new study created in memory with name: no-name-977639b9-1f75-4bf1-80dc-1ea1d7b4ef07
[I 2026-10-02 12:04:32,104] Trial 0 finished with value: -3280.7399824182694 and parameters: {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 25, 'max_features': 'log2'}. Best is trial 0 with value: -3280.7399824182694.
[I 2026-10-02 12:04:33,228] Trial 1 finished with value: -3190.489079913986 and parameters: {'n_estimators': 150, 'max_depth': 250, 'min_samples_split': 18, 'max_features': 'log2'}. Best is trial 1 with value: -3190.489079913986.
[I 2026-10-02 12:04:33,915] Trial 2 finished with value: -3281.741086003407 and parameters: {'n_estimators': 90, 'max_depth': 80, 'min_samples_split': 21, 'max_features': 'sqrt'}. Best is trial 1 with value: -3190.489079913986.
[I 2026-10-02 12:04:34,595] Trial 3 finished with value: -3258.917028049237 and parameters: {'n_estimators': 110, 'max_depth': 70, 'min_samples_split': 21, 'max_features': 'log2'}. Best is trial 1 

Best params: {'n_estimators': 120, 'max_depth': 100, 'min_samples_split': 24, 'max_features': None}


In [17]:
from numpy import mean, absolute

In [18]:
#Run the optimized model
regressor = RandomForestRegressor(**study.best_params,
                                      random_state=69) 
RFscores = cross_val_score(regressor, X, y, cv=5,
                           scoring='neg_mean_absolute_error')
MAE =mean(absolute(RFscores))
print ('the MAE is: %.2f' % MAE)

the MAE is: 2588.57


In [19]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Make the prediction
xgb = XGBClassifier(n_estimators=100, random_state=69, eval_metric='logloss')
y_pred = cross_val_predict(xgb, X, y, cv=5)  # make the predictions
cm = confusion_matrix(y, y_pred)  # create the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)  # create the graph
disp.plot()  # plot the graph
plt.show()  # show the graph
accuracy = (cm.diagonal().sum()) / cm.sum()  # calculate accuracy
print("Accuracy:", accuracy)  # print accuracy

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [   0    1    2 ... 1067 1068 1069], got [ 1121.8739   1131.5066   1135.9407  ... 60021.39897 62592.87309
 63770.42801]